<a href="https://colab.research.google.com/github/Foxokiso/hermes-agent/blob/main/HUNYUAN3D_PAINT_A100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hunyuan3D-Paint — texture an existing mesh (A100)

Opens **from GitHub**. Code clones **Tencent-Hunyuan/Hunyuan3D-2**. Weights come from Hugging Face.

**Drive is the file pipe** (not the code host):

| Role | Path |
|---|---|
| IN | `MyDrive/HUNYUAN/inputs/` — drop a `.glb`/`.obj`/`.ply` + a reference PNG/JPG |
| OUT | `MyDrive/HUNYUAN/outputs/` — textured GLB lands here |

This is **not TRELLIS**. TRELLIS makes a new mesh from a picture. This paints a mesh you already have.

Runtime: **Runtime → Change runtime type → A100 GPU + High-RAM**. Paint+delight wants ~16 GB. T4 is a maybe, not the run path.

License: **Tencent Hunyuan non-commercial**. Do not ship the outputs commercially.

**UV warning:** official `Hunyuan3DPaintPipeline` calls `mesh_uv_wrap` (xatlas). It **re-atlases**. Do **not** run this on Unity UV-locked bodies (Wicker / Disko / Magpie / HorseDing). Those stay local PBR. This notebook is for TRELLIS dumps and other throwaway meshes.


In [ ]:
# 0) GPU gate
import sys, torch
print("python", sys.version.split()[0])
assert torch.cuda.is_available(), "No CUDA. Runtime → Change runtime type → A100 GPU."
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
name = torch.cuda.get_device_name(0)
print(f"GPU: {name}  VRAM: {vram_gb:.1f} GB")
if vram_gb < 16:
    raise SystemExit(
        f"NEED >=16GB for paint. This box is {vram_gb:.1f}GB ({name}). "
        "Runtime → Change runtime type → A100."
    )
print("VRAM OK (A100 preferred; 16GB is the official paint floor).")


In [ ]:
# 1) Drive = IN + OUT. Code stays on GitHub. Weights stay on HF.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_IN = Path("/content/drive/MyDrive/HUNYUAN/inputs")
DRIVE_OUT = Path("/content/drive/MyDrive/HUNYUAN/outputs")
DRIVE_IN.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("IN :", DRIVE_IN)
print("    ", [p.name for p in DRIVE_IN.iterdir()][:20] or "(empty — drop a mesh + a ref image)")
print("OUT:", DRIVE_OUT)
print("    ", [p.name for p in DRIVE_OUT.iterdir()][:8])


In [ ]:
# 2) Clone official Tencent repo from GitHub (source of truth)
import os, subprocess
from pathlib import Path
os.chdir("/content")
dst = Path("/content/Hunyuan3D-2")
if dst.exists():
    print("already cloned")
else:
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git"],
        check=False,
    )
    if r.returncode != 0:
        raise SystemExit(f"git clone failed: {r.returncode}")
os.chdir("/content/Hunyuan3D-2")
sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("cloned Tencent-Hunyuan/Hunyuan3D-2 @", sha)


In [ ]:
# 3) Install. Keep Colab's torch. Fail the cell if a compile is nonzero.
import os, subprocess, sys
from pathlib import Path

os.chdir("/content/Hunyuan3D-2")
os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")

req = Path("requirements.txt").read_text(encoding="utf-8").splitlines()
keep = []
for line in req:
    s = line.strip()
    if not s or s.startswith("#"):
        continue
    name = s.split("==")[0].split(">=")[0].split("[")[0].strip().lower()
    if name in {"torch", "torchvision", "torchaudio"}:
        continue
    keep.append(s)
tmp = Path("/tmp/hy3d_req.txt")
tmp.write_text("\n".join(keep) + "\n", encoding="utf-8")

def run(cmd, cwd=None):
    print("+", " ".join(cmd) if isinstance(cmd, list) else cmd)
    r = subprocess.run(cmd, cwd=cwd)
    if r.returncode != 0:
        raise SystemExit(f"FAILED ({r.returncode}): {cmd}")

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(tmp)])
run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
run(
    [sys.executable, "setup.py", "install"],
    cwd="/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer",
)
run(
    [sys.executable, "setup.py", "install"],
    cwd="/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer",
)
print("install done")


In [ ]:
# 4) Load paint pipeline from Hugging Face (turbo). Not from Drive.
import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.insert(0, "/content/Hunyuan3D-2")

from hy3dgen.texgen import Hunyuan3DPaintPipeline

paint = Hunyuan3DPaintPipeline.from_pretrained(
    "tencent/Hunyuan3D-2",
    subfolder="hunyuan3d-paint-v2-0-turbo",
)
print("loaded tencent/Hunyuan3D-2  subfolder=hunyuan3d-paint-v2-0-turbo")


In [ ]:
# 5) Pick mesh + ref from Drive. Override names if more than one file is in inputs/.
from pathlib import Path
from PIL import Image

DRIVE_IN = Path("/content/drive/MyDrive/HUNYUAN/inputs")
MESH_NAME = ""   # e.g. "body.glb"  — empty = newest mesh
REF_NAME = ""    # e.g. "ref.png"   — empty = newest image

MESH_EXT = {".glb", ".gltf", ".obj", ".ply"}
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp"}

meshes = sorted(
    [p for p in DRIVE_IN.iterdir() if p.suffix.lower() in MESH_EXT],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
refs = sorted(
    [p for p in DRIVE_IN.iterdir() if p.suffix.lower() in IMG_EXT],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if MESH_NAME:
    mesh_path = DRIVE_IN / MESH_NAME
else:
    mesh_path = meshes[0] if meshes else None
if REF_NAME:
    ref_path = DRIVE_IN / REF_NAME
else:
    ref_path = refs[0] if refs else None

if mesh_path is None or not mesh_path.exists():
    raise SystemExit(
        "No mesh in MyDrive/HUNYUAN/inputs/. Put a .glb/.obj/.ply there and re-run this cell. "
        "FBX is not accepted — export GLB first."
    )
if ref_path is None or not ref_path.exists():
    raise SystemExit(
        "No reference image in MyDrive/HUNYUAN/inputs/. Put a PNG/JPG there and re-run this cell."
    )

img = Image.open(ref_path)
print("MESH", mesh_path, mesh_path.stat().st_size)
print("REF ", ref_path, img.size, img.mode)
img


In [ ]:
# 6) Paint. Writes textured GLB to Drive. Official pipeline re-UVs with xatlas.
from pathlib import Path
import time
import trimesh
from PIL import Image
from hy3dgen.rembg import BackgroundRemover

DRIVE_OUT = Path("/content/drive/MyDrive/HUNYUAN/outputs")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

mesh = trimesh.load(str(mesh_path), force="mesh")
if isinstance(mesh, trimesh.Scene):
    mesh = mesh.dump(concatenate=True)
print("faces", len(mesh.faces), "verts", len(mesh.vertices))

image = Image.open(ref_path).convert("RGBA")
if image.mode == "RGB":
    image = BackgroundRemover()(image)

t0 = time.time()
textured = paint(mesh, image=image)
print(f"paint {time.time()-t0:.1f}s")

stamp = time.strftime("%Y%m%d_%H%M%S")
out = DRIVE_OUT / f"textured_{mesh_path.stem}_{stamp}.glb"
textured.export(str(out))
print("DROPPED ON DRIVE")
print(out, out.stat().st_size)
